<a href="https://colab.research.google.com/github/simecek/dspracticum2026/blob/main/lesson07/01_tool_calling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 07 · Notebook 1: Tool calling and the agent loop

An LLM only writes text. It cannot look at the weather, run code or read your files. Yet ChatGPT searches the web, and coding agents like **opencode** edit files and run tests. How?

The trick is simple: we tell the model which **tools** (Python functions) exist. When it wants one, it writes a special piece of text, a **tool call**. *We* run the function and send the result back. Put this in a loop and you have an **agent**.

We do all of it by hand, with a tiny open model on Colab's GPU (free, no API key):

**Contents:** 0. A tiny model · 1. What the model cannot do · 2. A tool is just text · 3. Run the tool, send the result back · 4. The agent loop · 5. Pydantic: tools with types · 6. From here to opencode · 7. Summary

*Runtime → Change runtime type → T4 GPU.*

---
## 0. A tiny model

[Qwen3.5-0.8B](https://huggingface.co/Qwen/Qwen3.5-0.8B) is the smallest model of the Qwen3.5 family (Alibaba, 2026). It has 0.75 billion parameters (6× the smallest GPT-2 from Lesson 05, half of the largest one), and it was post-trained to call tools.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "Qwen/Qwen3.5-0.8B"
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16).to(device)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters on {device}")

*(You may see warnings about `causal_conv1d` or `flash-linear-attention`. Qwen3.5 mixes normal attention with a faster "linear attention", and these packages would only make it faster. Ignore them.)*

One helper: it turns a list of messages into the prompt (with the chat template, as in Lesson 05), generates the answer and returns it as text. `tools` and `think` are new, we will use them soon.

In [ ]:
END = tokenizer.convert_tokens_to_ids("<|im_end|>")   # the token that ends a message

def generate(messages, tools=None, think=False, max_new_tokens=500):
    inputs = tokenizer.apply_chat_template(messages, tools=tools, enable_thinking=think,
                                           add_generation_prompt=True, return_tensors="pt", return_dict=True).to(device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, eos_token_id=END)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print(generate([{"role": "user", "content": "Explain in one sentence what a transformer neural network is."}]))

---
## 1. What the model cannot do

In [ ]:
print(generate([{"role": "user", "content": "What is the weather in Brno right now?"}]))
print("-" * 80)
print(generate([{"role": "user", "content": "How many r's are in the word strawberry?"}]))

The model does not know today's weather: it learned from texts that are months old. And it miscounts letters (*strawberry* has three r's). Remember Lesson 06: the model sees tokens, not letters.

A few lines of Python can do both. So let's give the model Python functions.

---
## 2. A tool is just text

A tool is an ordinary Python function. The **docstring** matters: the model never sees the code, only the name, the description and the arguments.

In [ ]:
def get_weather(city: str) -> str:
    """Get the current weather in a city.

    Args:
        city: name of the city, e.g. Brno
    """
    fake_weather = {"brno": "12 °C, cloudy", "prague": "9 °C, rain", "vienna": "14 °C, sunny"}
    return fake_weather.get(city.lower(), "unknown city")

get_weather("Brno")

*(The weather is made up. A real tool would call a weather API, but the idea is the same.)*

**Look inside.** We pass the function to the chat template and print the prompt the model actually gets:

In [ ]:
messages = [{"role": "user", "content": "What is the weather in Brno right now?"}]
prompt = tokenizer.apply_chat_template(messages, tools=[get_weather], add_generation_prompt=True, tokenize=False)
print(prompt)

There is no magic:
- `transformers` turned the function into a **JSON schema** (name, description, the type of each argument) and pasted it into a **system message**, together with instructions on how to call it.
- During post-training (Lesson 06), the model saw many conversations with tools in exactly this format, and learned when and how to use them.

Every tool costs tokens in every request:

In [ ]:
print(len(tokenizer(prompt)["input_ids"]), "tokens with the tool")
print(len(tokenizer(tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False))["input_ids"]), "tokens without it")

Now let the model answer:

In [ ]:
answer = generate(messages, tools=[get_weather])
print(answer)

The model did **not** answer the question. It wrote a **tool call**: which function to call and with which arguments. Nothing has been run yet. The model cannot run anything, it only writes text.

---
## 3. Run the tool, send the result back

Three steps: **parse** the tool call from the text, **run** the function, **send** the result back to the model.

**Parse.** Two regular expressions are enough: one for `<function=...>...</function>`, one for each `<parameter=...>`.

In [ ]:
import re

def parse_tool_calls(text):
    calls = []
    for name, body in re.findall(r"<function=(\w+)>(.*?)</function>", text, re.DOTALL):
        args = {key: value.strip() for key, value in re.findall(r"<parameter=(\w+)>(.*?)</parameter>", body, re.DOTALL)}
        calls.append({"name": name, "arguments": args})
    return calls

calls = parse_tool_calls(answer)
calls

**Run.** We look up the function by its name and call it with the arguments:

In [ ]:
TOOLS = {"get_weather": get_weather}

call = calls[0]
result = TOOLS[call["name"]](**call["arguments"])
result

**Send back.** We append two messages to the conversation: the model's tool call (role `assistant`), and the result (role `tool`). Then we ask the model again:

In [ ]:
messages.append({"role": "assistant", "content": "", "tool_calls": [{"type": "function", "function": call}]})
messages.append({"role": "tool", "name": call["name"], "content": result})

print(generate(messages, tools=[get_weather]))

**Look inside.** How does the result look in the prompt? Here are its last lines:

In [ ]:
print(tokenizer.apply_chat_template(messages, tools=[get_weather], add_generation_prompt=True, tokenize=False)[-300:])

The result is just text inside `<tool_response>`. The model trusts it. If a tool returns a web page with "ignore your instructions and…", the model reads that too. This is called **prompt injection**, and it is the biggest security problem of agents.

**Try it:** Ask about a city that `get_weather` does not know. What does the model say?

---
## 4. The agent loop

Sometimes one tool call is not enough. The model may need several calls, or call one tool, look at the result and then call another. So we repeat the same steps **in a loop**, until the model answers with text instead of a tool call:

```
question → model → tool call? ─yes→ run the tools → add the results → model → ...
                       └─no──→ final answer
```

This loop **is** an agent. We add two more tools first:

In [ ]:
def count_letter(word: str, letter: str) -> int:
    """Count how many times a letter occurs in a word.

    Args:
        word: the word
        letter: a single letter
    """
    return word.lower().count(letter.lower())

def multiply(a: float, b: float) -> float:
    """Multiply two numbers.

    Args:
        a: the first number
        b: the second number
    """
    return a * b

TOOLS = {f.__name__: f for f in [get_weather, count_letter, multiply]}
list(TOOLS)

In [ ]:
def run_agent(question, system=None, think=False, max_steps=5):
    messages = [{"role": "system", "content": system}] if system else []
    messages.append({"role": "user", "content": question})

    for step in range(max_steps):
        answer = generate(messages, tools=list(TOOLS.values()), think=think)
        if "</think>" in answer:                        # with think=True, show the reasoning separately
            thought, answer = answer.split("</think>", 1)
            print(f"💭 {thought.strip()}")
            answer = answer.strip()
        calls = parse_tool_calls(answer)
        if not calls:                                   # no tool call = the final answer
            print(f"🤖 {answer}")
            return messages

        messages.append({"role": "assistant", "content": "", "tool_calls": [{"type": "function", "function": c} for c in calls]})
        for call in calls:
            try:
                result = TOOLS[call["name"]](**call["arguments"])
            except Exception as e:                      # errors go back to the model, too
                result = f"Error: {e!r}"
            print(f"🔧 step {step}: {call['name']}({call['arguments']}) → {result}")
            messages.append({"role": "tool", "name": call["name"], "content": str(result)})

    print("⚠️ too many steps, giving up")
    return messages

`max_steps` matters: a confused agent can loop forever, and with a paid API every step costs money.

In [ ]:
run_agent("What is the weather in Brno and in Prague?");

Two tool calls in one step: the model asked for both cities at once (*parallel tool calls*).

Now strawberry again:

In [ ]:
run_agent("How many r's are in strawberry?");

The model had the tool, but it answered by itself, and got it wrong again. It has to decide *when* to use a tool, and a 0.8B model is not good at deciding. Two ways to help it:

In [ ]:
run_agent("How many r's are in strawberry?",
          system="Always use a tool for counting letters and for arithmetic.")
print("-" * 80)
run_agent("How many r's are in strawberry?", think=True);

With `think=True`, the model first writes its reasoning (💭, the text before `</think>`, Lesson 06), and there it decides to use the tool. This is what "models think with tools" means.

**Try it:** `run_agent("How many r's are in strawberry, multiplied by 7?")`. Does the model use both tools? Try a few more questions. Where does the tiny model fail?

---
## 5. Pydantic: tools with types

Let's multiply:

In [ ]:
run_agent("What is 123.5 times 4.2?");

The model even recovered by itself: it read the error and did the multiplication in its head. But look at the 🔧 line: `multiply` got `'123.5'` and `'4.2'`, which are **strings**, not numbers. The model only writes text, and our parser does not know that `a` should be a `float`. Python's `*` then fails (or, for `'3' * 2`, it quietly returns `'33'`!).

Arguments written by the model need to be **checked and converted**: Is every required argument there? Is it a number? Is the letter really one letter? This is what **pydantic** does. We used it in Lesson 05 for structured output (`ReviewRating`). Here it describes the arguments of a tool:

In [ ]:
from pydantic import BaseModel, Field, ValidationError

class Multiply(BaseModel):
    """Multiply two numbers."""
    a: float = Field(description="the first number")
    b: float = Field(description="the second number")

class CountLetter(BaseModel):
    """Count how many times a letter occurs in a word."""
    word: str = Field(description="the word")
    letter: str = Field(description="a single letter", min_length=1, max_length=1)

**1. Validate and convert.** `model_validate` turns the strings into numbers:

In [ ]:
args = Multiply.model_validate({"a": "123.5", "b": "4.2"})
print(args, "→", args.a * args.b)

…and when an argument is wrong, it says exactly what is wrong:

In [ ]:
try:
    CountLetter.model_validate({"word": "strawberry", "letter": "rr"})
except ValidationError as e:
    print(e)

This error message is great for the agent: we send it back as the tool result, and the model can correct itself in the next step.

**2. Generate the schema.** Pydantic writes the JSON schema for us. It is the same schema `transformers` made from the docstring in section 2, and the same one the OpenAI API needs:

In [ ]:
import json
print(json.dumps(Multiply.model_json_schema(), indent=2))

A small helper turns a pydantic class into a tool definition, in the format that `apply_chat_template` and the OpenAI API both accept:

In [ ]:
def tool_schema(name, args_model):
    schema = args_model.model_json_schema()
    schema.pop("title"); schema.pop("description", None)      # these two are already in name and description
    return {"type": "function",
            "function": {"name": name, "description": args_model.__doc__, "parameters": schema}}

tool_schema("multiply", Multiply)

Now each tool is a pair: a pydantic class (the arguments) and a function (what it does). The loop validates every call before running it:

In [ ]:
TYPED_TOOLS = {
    "multiply":     (Multiply,    lambda args: args.a * args.b),
    "count_letter": (CountLetter, lambda args: args.word.lower().count(args.letter.lower())),
}

def run_typed_agent(question, system=None, think=False, max_steps=5):
    schemas = [tool_schema(name, args_model) for name, (args_model, _) in TYPED_TOOLS.items()]
    messages = [{"role": "system", "content": system}] if system else []
    messages.append({"role": "user", "content": question})

    for step in range(max_steps):
        answer = generate(messages, tools=schemas, think=think)
        if "</think>" in answer:
            thought, answer = answer.split("</think>", 1)
            print(f"💭 {thought.strip()}")
            answer = answer.strip()
        calls = parse_tool_calls(answer)
        if not calls:
            print(f"🤖 {answer}")
            return messages

        messages.append({"role": "assistant", "content": "", "tool_calls": [{"type": "function", "function": c} for c in calls]})
        for call in calls:
            if call["name"] not in TYPED_TOOLS:
                result = f"Error: unknown tool {call['name']}"
            else:
                args_model, function = TYPED_TOOLS[call["name"]]
                try:
                    result = function(args_model.model_validate(call["arguments"]))   # ← validate, then run
                except ValidationError as e:
                    result = f"Error: {e}"
            print(f"🔧 step {step}: {call['name']}({call['arguments']}) → {result}")
            messages.append({"role": "tool", "name": call["name"], "content": str(result)})

    print("⚠️ too many steps, giving up")
    return messages

run_typed_agent("What is 123.5 times 4.2?");

This is how real libraries work. With the OpenAI API (Lesson 05), the same pydantic class becomes a tool in one line, and the API even guarantees that the arguments match the schema:

```python
import openai
tools = [openai.pydantic_function_tool(Multiply)]
response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
response.choices[0].message.tool_calls      # arguments as JSON, ready for Multiply.model_validate_json(...)
```

Agent frameworks (OpenAI Agents SDK, PydanticAI, smolagents, LangGraph...) wrap exactly this loop: schemas from types, validation, the loop and `max_steps`.

**Try it:** Add a tool `convert_currency(amount, from_currency, to_currency)` with a pydantic class (hint: `Literal["CZK", "EUR", "USD"]` restricts the currency). Use made-up exchange rates.

---
## 6. From here to opencode

A **coding agent** like [opencode](https://opencode.ai) is this same loop, with different tools:

| tool | what it does |
|---|---|
| `read`, `glob`, `grep` | read files, find files, search in files |
| `edit`, `write` | change a file, create a file |
| `shell` | run a command: `python train.py`, `pytest`, `git diff`... |
| `webfetch`, `websearch` | read a web page, search the web |

Plus a long system prompt with the rules, and `AGENTS.md`, a file in your project that tells the agent about it (how to run tests, which style to use).

With `shell`, the agent can run *any* command on your computer. That's why opencode asks for **permission** before risky actions: you decide what it may do alone, and what it must ask about first.

And the model? A 0.8B model can call a tool or two, but it gets lost quickly (try `run_agent("What is 17 times 23?")` a few times). Coding agents need much bigger models, which run through an API. We switch to them in the next part.

---
## 7. Summary

- The model only **writes text**. A tool call is text in an agreed format; **we** parse it, run the function, and send the result back as a `tool` message.
- The tool descriptions are pasted into the prompt, so they cost tokens in every request. The model sees the name, the description and the arguments, never the code.
- An **agent** is a loop: model → tool calls → results → model → ... until it answers with text. Always set `max_steps`.
- **Pydantic** describes the arguments: it generates the JSON schema and validates and converts what the model wrote. Validation errors go back to the model.
- Tool results are untrusted text (**prompt injection**). An agent with a shell needs permissions.